In [29]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, avg, count, when, sum
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from snowflake.ml.data.data_connector import DataConnector

from snowflake.snowpark import Session
from credentials import params

from snowflake.snowpark.types import (
    IntegerType,
    LongType,
    FloatType,
    DoubleType,
    DecimalType,
    StringType
)

from snowflake.ml.registry import Registry

session = Session.builder.configs(params).create()

In [30]:
#df_sp = session.table("HOUSING_PRICE_PROJECT.STAGING_LAYER.RAW_DATA")
#df = df_sp.to_pandas()

X_train_sp = session.table("HOUSING_PRICE_PROJECT.ML_LAYER.TRAIN_SET")
X_train_pd = X_train_sp.to_pandas()

#X_test_sp = session.table("HOUSING_PRICE_PROJECT.ML_LAYER.TEST_SET")
#X_test = X_test_sp.to_pandas()

In [31]:
session.use_database("HOUSING_PRICE_PROJECT")
session.use_schema("ML_LAYER")

# Hyperparameter Tuning

In [33]:
from snowflake.ml.jobs import remote

from ML_hyperp_tune_func1 import dataset_map_creator
from ML_hyperp_tune_func2 import best_hyperparam

In [34]:
session.use_database("HOUSING_PRICE_PROJECT")
session.use_schema("ML_LAYER")

@remote(
    "ml_job_computerpool",
    stage_name="HOUSING_PRICE_PROJECT.ML_LAYER.MLJOBS_STAGE",
    session=session,
    target_instances=4,
    imports=[
        "ML_hyperp_tune_func1.py",
        "ML_hyperp_tune_func2.py"
    ]
)
def hyperparam_tuning(train_name):
    from ML_hyperp_tune_func1 import dataset_map_creator
    from ML_hyperp_tune_func2 import best_hyperparam
    
    from ray import tune
    from snowflake.ml.data.data_connector import DataConnector
    from snowflake.ml.modeling.distributors.xgboost.xgboost_estimator import XGBScalingConfig
    from snowflake.ml.runtime_cluster import scale_cluster
    from snowflake.ml.modeling.tune import get_tuner_context
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    from snowflake.ml.modeling.distributors.xgboost.xgboost_estimator import XGBEstimator   
    from snowflake.ml.modeling.tune import (Tuner, TunerConfig, get_tuner_context) 
    from snowflake.snowpark.types import StringType


    # SESSION, DATASET AND FEATURE LIST ************************************************************************
    job_session = get_active_session()
    print("librerie importate + session activated *********************************")
 
    train_tab = job_session.table(train_name)
    #train_tab = train_tab.drop('"property_id"', '"price_category"', 'SOURCE_FILE') # <--- REMOVED ON 29/08
    
    #train_tab = X_train
    
    #for column in train_tab.columns:                                               # <--- REMOVED ON 29/08
    #    train_tab = train_tab.with_column_renamed(column, col.replace('"', ''))
    
    print("train_tab importato *********************************")
    
    
    # DATASET_MAP CREATION USING EXTERNAL FUNCTION **************************************************************
    # PREPROCESSING HAPPENS WITHIN THIS FUNCTION !! *************************************************************
    dataset_map, features, preproc_model_version = dataset_map_creator(job_session, train_tab, size_tuning_df = 15_000, k_cv = 3) 
                                                  #dataset_map_creator(train_tab, job_session)
                                
    print("dataset_map e features ritornate *********************************")

    #CREATING FEATURE LIST **************************************************************************************
    #features = train_tab_prepr.columns
    #features.remove('PRICE_IN_LAKHS')
    # FEATURES RETURNED BY dataset_map_creator()


    

    # SEARCH SPACE *********************************************************************************************
    search_space = {
        "max_depth": tune.choice([3]), #[3, 6, 9], during development it was = 2
        "learning_rate": tune.choice([0.1]), #[0.01, 0.05, 0.1], , during development it was = 0.05
        "n_estimators": tune.choice([4]), #[100, 200, 300] , during development it was = 3
    }
    print("search space defined *********************************")

    #TUNER CONFIG **********************************************************************************************
    tuner_config = TunerConfig(
        metric="rmse_valid_avg",
        mode="min",
        num_trials=1,
        uses_snowflake_trainer=True,
        max_concurrent_trials=5
    )
    print("tuner config defined *********************************")

    #TRAIN FUNC DEFINITION **************************************************************************************
    def train_func():
    
        ctx = get_tuner_context()
    
        hyper_params = ctx.get_hyper_params()
        datasets = ctx.get_dataset_map()
    
        print("hyperparam and dataset retrieved from context *********************************")

        print(f"hyper_params[max_depth] = {hyper_params["max_depth"]}, type = {type(hyper_params["max_depth"])}")
        print(f"hyper_params[learning_rate] = {hyper_params["learning_rate"]}, type = {type(hyper_params["learning_rate"])}")
        print(f"hyper_params[n_estimators] = {hyper_params["n_estimators"]}, type = {type(hyper_params["n_estimators"])}")
        
        ###################################################################################################################

        rmse_train_list = []
        mae_train_list = []
        r2_train_list = []
        
        rmse_valid_list = []
        mae_valid_list = []
        r2_valid_list = []
        
        for i in range(int(len(datasets)/2)):
            train_connector = datasets[f"train_{i}"]
            valid_connector = datasets[f"valid_{i}"]
        
            estimator = XGBEstimator(
                params={
                    "max_depth": hyper_params["max_depth"],
                    "learning_rate": hyper_params["learning_rate"],
                },
                n_estimators=hyper_params["n_estimators"],
                objective="reg:squarederror",
                scaling_config=XGBScalingConfig(
                    num_workers=1
                )
            )
            print(f"estimator created, ciclo {i} *********************************")

            
            estimator.fit(
                dataset=train_connector,
                input_cols=features,
                label_col='PRICE_IN_LAKHS'
            )
    
            print(f"estimator trained, ciclo {i} *********************************")

            
            train_conn_pd = train_connector.to_pandas()
            X_train_pd = train_conn_pd.drop(['PRICE_IN_LAKHS'], axis = 1)
            target_train = train_conn_pd['PRICE_IN_LAKHS']

            valid_conn_pd = valid_connector.to_pandas()
            X_valid_pd = valid_conn_pd.drop(['PRICE_IN_LAKHS'], axis = 1)
            target_valid = valid_conn_pd['PRICE_IN_LAKHS']
    
            print(f"pandas dataframe created, ciclo {i} *********************************")
    
            predictions_train = estimator.predict(X_train_pd)
               
            rmse_train = mean_squared_error(
                y_true = target_train,
                y_pred = predictions_train
            )

            mae_train = mean_absolute_error(
                y_true = target_train,
                y_pred = predictions_train
            )
        
            r2_train = r2_score(
                y_true = target_train,
                y_pred = predictions_train
            )
            
            rmse_train_list.append(rmse_train)
            mae_train_list.append(mae_train)
            r2_train_list.append(r2_train)
            
            print(f"rmse_train_{i} = {rmse_train} *********************************")
            
            predictions_valid = estimator.predict(X_valid_pd)
               
            rmse_valid = mean_squared_error(
                y_true = target_valid,
                y_pred = predictions_valid
            )

            mae_valid = mean_absolute_error(
                y_true = target_valid,
                y_pred = predictions_valid
            )
        
            r2_valid = r2_score(
                y_true = target_valid,
                y_pred = predictions_valid
            )

            rmse_valid_list.append(rmse_valid)
            mae_valid_list.append(mae_valid)
            r2_valid_list.append(r2_valid)
            
            print(f"rmse_valid_{i} = {rmse_valid} *********************************")
        

        print("\nrmse_train (average) = ", np.mean(rmse_train_list))
        print("\nrmse_valid (average) = ", np.mean(rmse_valid_list))

        ###################################################################################################################
        
        rmse_train_avg = np.mean(rmse_train_list)
        rmse_valid_avg = np.mean(rmse_valid_list)

        mae_train_avg = np.mean(mae_train_list)
        mae_valid_avg = np.mean(mae_valid_list)

        r2_train_avg = np.mean(r2_train_list)
        r2_valid_avg = np.mean(r2_valid_list)
        
        ctx.report(
            metrics={
                "rmse_valid": rmse_valid_list, 
                "rmse_train": rmse_train_list,
                "rmse_train_avg":  rmse_train_avg,
                "rmse_valid_avg": rmse_valid_avg,
                "mae_train_avg": mae_train_avg,
                "mae_valid_avg": mae_valid_avg,
                "r2_train_avg": r2_train_avg,
                "r2_valid_avg": r2_valid_avg
            },
            model=estimator
        )
        print("\nctx.report ok *********************************")

    #TUNER DECLARATION ******************************************************************************************
    tuner = Tuner(
        train_func=train_func,
        search_space=search_space,
        tuner_config=tuner_config
    )
    print("\ntuner declaration done *********************************")
    
    #RUN THE TUNER ******************************************************************************************     

    results = tuner.run(dataset_map = dataset_map)
        
    print("\nsembrerebbe essere arrivato fino alla fine *********************************")

    res = results.results
    
    print(type(res), end = "\n\n")
    res_cols = list(res.columns)
    print(res_cols, end = "\n\n")
    print(res.info(), end = "\n\n")
    print(res["rmse_train"], end = "\n\n")
    print(res["rmse_valid"], end = "\n\n")

    #best_hyperparam(res, preproc_model_version, job_session)
    best_hyperparam(job_session, res, preproc_model_version, gap_max = 100)
    

In [35]:
import time
start = time.time()

job = hyperparam_tuning("HOUSING_PRICE_PROJECT.ML_LAYER.TRAIN_SET") 
job.wait()

end = time.time()

print(job.status)
print(f"{(end - start) / 60 :.2f} min")

DONE
1.73 min


In [36]:
job.show_logs(verbose=True)

2026-08-29T18:47:40.161Z	info	otelconftelemetry/tracer.go:47	Internal trace telemetry disabled	{"resource": {"service.instance.id": "c5f610d5-4c5f-4e40-90ff-e5f281f91224", "service.name": "otelcol-contrib", "service.version": "0.156.0"}}
2026-08-29T18:47:40.162Z	warn	builders/builders.go:40	"otlp" alias is deprecated; use "otlp_grpc" instead	{"resource": {"service.instance.id": "c5f610d5-4c5f-4e40-90ff-e5f281f91224", "service.name": "otelcol-contrib", "service.version": "0.156.0"}, "otelcol.component.id": "otlp", "otelcol.component.kind": "exporter", "otelcol.signal": "metrics"}
2026-08-29T18:47:40.180Z	info	service@v0.156.0/service.go:256	Starting otelcol-contrib...	{"resource": {"service.instance.id": "c5f610d5-4c5f-4e40-90ff-e5f281f91224", "service.name": "otelcol-contrib", "service.version": "0.156.0"}, "Version": "0.156.0", "NumCPU": 8}
2026-08-29T18:47:40.180Z	info	extensions/extensions.go:41	Starting extensions...	{"resource": {"service.instance.id": "c5f610d5-4c5f-4e40-90ff-e5f

In [37]:
job.status

'DONE'

# Tuner

In [ ]:
def dataset_map_creator(job_session, data_table, size_tuning_df = 15_000, k_cv = 3)

    from snowflake.snowpark.functions import row_number, random, col
    from snowflake.snowpark.window import Window
    from snowflake.ml.data.data_connector import DataConnector
    
    SIZE_TUNING_DF = size_tuning_df
    K_CV = k_cv
    
    # import the dataset as snowpark dataframe
    df_sp = data_table#job_session.table(data_table)
    
    # take a random sample of SIZE_TUNING_DF rows
    tuning_df = df_sp.sample(n = SIZE_TUNING_DF)
    
    # updating SIZE_TUNING_DF in case df_sp.count() < SIZE_TUNING_DF
    SIZE_TUNING_DF = tuning_df.count()
    
    # save tuning_df as temporary table to avoid that any time tuning_df is called new random rows are taken
    tuning_df.write.mode("overwrite").save_as_table(
        "TEMP_TUNING_DATA",
        table_type="temporary"
    )
    
    # importing tuning_df from the temporary tab
    tuning_tab = job_session.table("TEMP_TUNING_DATA")

    print("tutta la prima parte di tuner() e' ok *********************************")

    
    
    # PREPROCESSING using the pipeline from model registry *******************************************************
    registry = Registry(
        session=job_session,
        database_name="HOUSING_PRICE_PROJECT",
        schema_name="ML_LAYER"
    )
    
    preprocessor_model = registry.get_model("PREPROCESSING_PIPELINE").version("LAST")    
    
    tuning_tab_prepr = preprocessor_model.run(
        tuning_tab,
        function_name="transform"
    )    
 
    # ADJUSTING THE PREPROCESSED DATASET *************************************************************************
    for column in tuning_tab_prepr.columns:
        #if column[0] =='"':
        tuning_tab_prepr = tuning_tab_prepr.with_column_renamed(column, column.replace('"', ''))
    
    cat_cols = [
        field.name
        for field in tuning_tab_prepr.schema.fields
        if isinstance(field.datatype, StringType)    
    ]
    
    tuning_tab_prepr = tuning_tab_prepr.drop(cat_cols)


    # CREATE FEATURE LIST
    features = tuning_tab_prepr.columns
    print(type(features))
    print()
    print(features)
    features.remove('PRICE_IN_LAKHS')

    print("preprocessing, dataset adjustment e feature list creation done! *********************************")


    
    
    # creating "random_numb" column to numerate each row
    window = Window.order_by(random())
    tuning_tab_prepr = tuning_tab_prepr.with_column(
        "RANDOM_NUMB",
        row_number().over(window)
    )
    print("tuner 1 *********************************")
    
    # creating the ranges list, containing ranges on which to filter the dataframe; example: ranges = [(0, 100), (100, 200), ...]
    i = 0
    frac = SIZE_TUNING_DF / K_CV
    
    ranges = []
    for n in range(K_CV):
        j = (i, i + frac)
        ranges.append(j)
        i += frac
    print("tuner 2 *********************************")
    
    # separating the snowpark dataframe in the k folds for the CV
    i = 0
    folds = []

    print(ranges)
    print(type(tuning_tab_prepr))
    
    for low_limit, up_limit in ranges:
        fold = tuning_tab_prepr.filter((col("RANDOM_NUMB") > low_limit) & (col("RANDOM_NUMB") <= up_limit))
        folds.append(fold)
        i += 1
        
    print("tuner 3 *********************************")
    # creating dataset_map with proper fold combinations
    dataset_map = {}

    for validation_idx in range(K_CV):
    
        validation_df = folds[validation_idx]
    
        train_df = None
    
        for i, fold in enumerate(folds):
            if i != validation_idx:
                if train_df is None:
                    train_df = fold
                else:
                    train_df = train_df.union(fold)
        
        dataset_map[f"train_{validation_idx}"] = DataConnector.from_dataframe(train_df.drop("RANDOM_NUMB"))
        dataset_map[f"valid_{validation_idx}"] = DataConnector.from_dataframe(validation_df.drop("RANDOM_NUMB"))
    print("tuner 4 *********************************")
    
    return dataset_map, features, preprocessor_model.version_name

In [ ]:
X_train = session.table("HOUSING_PRICE_PROJECT.ML_LAYER.TRAIN_SET")
#X_train = X_train.drop('"property_id"', '"price_category"', 'SOURCE_FILE')

#for column in X_train.columns:
#    if column[0] =='"':
#        X_train = X_train.with_column_renamed(column, column[1:-1])

In [ ]:
a, b = dataset_map_creator(X_train, session)

In [ ]:
a

In [ ]:
a['train_0'].to_pandas()[['PROPERTY_ID', 'RANDOM_NUMB']].sort_values(by= 'RANDOM_NUMB')

In [ ]:
a['valid_0'].to_pandas()[['PROPERTY_ID', 'RANDOM_NUMB']].sort_values(by= 'RANDOM_NUMB')

# how to choose the best model

In [ ]:
import pandas as pd

res = pd.DataFrame({
    "rmse_train": [
        [7956.8466796875, 7889.2763671875, 8068.8154296875],
        [7856.8466796875, 7789.2763671875, 7968.8154296875],
        [7756.8466796875, 7789.2763671875, 7996.8154296875]
    ],

    "rmse_valid": [
        [9521.8466796875, 9487.2763671875, 9602.8154296875],
        [9856.8466796875, 9789.2763671875, 9468.8154296875],
        [9956.8466796875, 9889.2763671875, 9568.8154296875]
    ],

    "should_checkpoint": [
        True,
        True,
        True
    ],

    "trial_id": [
        "996a3_00000",
        "996a3_00000",
        "996a3_00000"
    ],

    "time_total_s": [
        126.91735076904297,
        126.91735076904297,
        126.91735076904297
    ],

    "config/max_depth": [
        1,
        2,
        3
    ],
    
    "config/learning_rate": [
        0.05,
        0.06,
        0.07
    ],

    "config/n_estimators": [
        3,
        4,
        5
    ],

    "config/min_child_weight": [
        1,
        5,
        10
    ],

    "config/subsample": [
        0.8,
        1.0,
        1.2
    ],    
})

In [ ]:
def best_hyperparam(job_session, res, preproc_model_version, gap_max = 100):
    
    from datetime import datetime
    GAP_MAX = gap_max
    
    # RENAME COLUMNS TO REMOVE "config/" ***************************************************************************
    res.rename(columns = { col : col.replace("config/", "") for col in res.columns }, inplace = True)

    # CREATE NEW COLUMNS AND CALCULATE AVERAGE METRICS **************************************************************
    timestamp = datetime.now()
    res["timestamp"] = timestamp
 
    timestamp_code = timestamp.strftime("%Y%m%d_%H%M%S")
    res["tuning_run_id"] = timestamp_code

    res["preprocessor_version"] = preproc_model_version
    
    #res["rmse_train_avg"] = res["rmse_train"].apply(lambda x: np.mean(x))
    #res["rmse_valid_avg"] = res["rmse_valid"].apply(lambda x: np.mean(x))
    
    res["relative_gap"] = (res["rmse_valid_avg"] - res["rmse_train_avg"])/res["rmse_train_avg"]

    
    # FIND THE RAW WITH BEST METRICS **************************************************************
    rmse_valid_avg_min = res.loc[res.relative_gap < GAP_MAX, "rmse_valid_avg"].min()
    best_raw = res[res["rmse_valid_avg"] == rmse_valid_avg_min]

    print("arriva fino a qui\n")
    print(res.columns)
    

    for data, tab in [(res, "XGBOOST_TUNING_RESULTS"), (best_raw, "XGBOOST_BEST_PARAMETERS")]:
        
        snowpark_df = job_session.create_dataframe(data)

        snowpark_df.write.mode("overwrite").save_as_table(
            "results_to_append",
            table_type="temporary"
        )
        
        print("\nsnowpark_df.columns:\n", snowpark_df.columns)
        
        query = f"""
            INSERT INTO {tab} ({", ".join(list(data.columns))})
            SELECT * FROM RESULTS_TO_APPEND
        """
        
        print("\nquery:\n", query)
        
        job_session.sql(query).collect()

In [22]:
session.sql("""
    CREATE OR REPLACE TABLE XGBOOST_BEST_PARAMETERS (
        tuning_run_id VARCHAR,
        timestamp TIMESTAMP_NTZ,
        
        rmse_train_avg DOUBLE,
        rmse_valid_avg DOUBLE,
        relative_gap DOUBLE, 

        
        max_depth BIGINT DEFAULT 6,
        learning_rate DOUBLE DEFAULT 0.3,
        n_estimators BIGINT DEFAULT 100,
        min_child_weight BIGINT DEFAULT 1,
        subsample DOUBLE DEFAULT 1.0,

        mae_train_avg DOUBLE,
        mae_valid_avg DOUBLE,
        r2_train_avg DOUBLE,
        r2_valid_avg DOUBLE,
        rmse_train VARIANT,
        rmse_valid VARIANT,

        preprocessor_version VARCHAR, 
        should_checkpoint BOOLEAN,
        trial_id VARCHAR,
        time_total_s DOUBLE
    );
""").collect()

[Row(status='Table XGBOOST_BEST_PARAMETERS successfully created.')]

In [23]:
session.sql("""
    CREATE OR REPLACE TABLE XGBOOST_TUNING_RESULTS (
        tuning_run_id VARCHAR,
        timestamp TIMESTAMP_NTZ,
        
        rmse_train_avg DOUBLE,
        rmse_valid_avg DOUBLE,
        relative_gap DOUBLE, 

        
        max_depth BIGINT DEFAULT 6,
        learning_rate DOUBLE DEFAULT 0.3,
        n_estimators BIGINT DEFAULT 100,
        min_child_weight BIGINT DEFAULT 1,
        subsample DOUBLE DEFAULT 1.0,

        mae_train_avg DOUBLE,
        mae_valid_avg DOUBLE,
        r2_train_avg DOUBLE,
        r2_valid_avg DOUBLE,
        rmse_train VARIANT,
        rmse_valid VARIANT,

        preprocessor_version VARCHAR,
        should_checkpoint BOOLEAN,
        trial_id VARCHAR,
        time_total_s DOUBLE
    );
""").collect()

[Row(status='Table XGBOOST_TUNING_RESULTS successfully created.')]

In [24]:
#best_hyperparam(res, session)

In [27]:
session.sql("""SELECT * FROM XGBOOST_TUNING_RESULTS""").to_pandas()

,TUNING_RUN_ID,TIMESTAMP,RMSE_TRAIN_AVG,RMSE_VALID_AVG,RELATIVE_GAP,MAX_DEPTH,LEARNING_RATE,N_ESTIMATORS,MIN_CHILD_WEIGHT,SUBSAMPLE,MAE_TRAIN_AVG,MAE_VALID_AVG,R2_TRAIN_AVG,R2_VALID_AVG,RMSE_TRAIN,RMSE_VALID,PREPROCESSOR_VERSION,SHOULD_CHECKPOINT,TRIAL_ID,TIME_TOTAL_S


In [28]:
session.sql("""SELECT * FROM XGBOOST_BEST_PARAMETERS""").to_pandas()

,TUNING_RUN_ID,TIMESTAMP,RMSE_TRAIN_AVG,RMSE_VALID_AVG,RELATIVE_GAP,MAX_DEPTH,LEARNING_RATE,N_ESTIMATORS,MIN_CHILD_WEIGHT,SUBSAMPLE,MAE_TRAIN_AVG,MAE_VALID_AVG,R2_TRAIN_AVG,R2_VALID_AVG,RMSE_TRAIN,RMSE_VALID,PREPROCESSOR_VERSION,SHOULD_CHECKPOINT,TRIAL_ID,TIME_TOTAL_S
